# Analisis Segmentasi Profitabilitas dan Efektivitas Diskon
## Menggunakan Metode K-Means Clustering

Notebook ini berisi implementasi analisis dataset Superstore dari tahap pemuatan data sampai ekspor hasil clustering.

## 1. Import Library

Library digunakan untuk membaca data, preprocessing, standardisasi, dan clustering.

In [ ]:
import pandas as pd
from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.stats import kruskal, spearmanr, chi2_contingency

## 2. Upload dan Membaca Dataset

Unggah file `soal2_Superstore.csv` dari komputer ke Google Colab.

In [ ]:
uploaded = files.upload()
df = pd.read_csv("soal2_Superstore.csv")
df_awal = df.copy()
print("Dimensi awal:", df.shape)
df.head()

## 3. Eksplorasi Kualitas Data

Pemeriksaan meliputi tipe data, missing value, dan data duplikat.

In [ ]:
print(df.info())
print("\\nMissing value:")
print(df.isna().sum())
print("\\nJumlah duplikat:", df.duplicated().sum())

## 4. Menghapus Data Duplikat

Baris duplikat dihapus agar transaksi yang sama tidak dihitung berulang.

In [ ]:
df = df.drop_duplicates().copy()
print("Dimensi setelah deduplikasi:", df.shape)

## 5. Menentukan Fitur Analisis

Empat fitur numerik digunakan untuk mengukur nilai dan profitabilitas transaksi. Dua fitur kategorikal digunakan untuk memasukkan konteks produk dan wilayah.

In [ ]:
fitur_numerik = ["Sales", "Quantity", "Discount", "Profit"]
fitur_kategorikal = ["Category", "Region"]
fitur = fitur_numerik + fitur_kategorikal
df[fitur].head()

## 6. Preprocessing Data

Missing value numerik diisi dengan median, missing value kategorikal diisi dengan modus, kategori diubah dengan one-hot encoding, dan fitur numerik distandardisasi.

In [ ]:
preprocessor = ColumnTransformer([
    ("numerik", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), fitur_numerik),
    ("kategorikal", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), fitur_kategorikal)
])

X = preprocessor.fit_transform(df[fitur])
print("Dimensi data hasil preprocessing:", X.shape)

## 7. Membentuk Model K-Means

Model menggunakan empat cluster. `random_state` digunakan agar hasil dapat direproduksi.

In [ ]:
model = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Cluster"] = model.fit_predict(X)
print(df["Cluster"].value_counts().sort_index())

## 8. Profil dan Interpretasi Cluster

Rata-rata fitur numerik digunakan untuk memahami karakteristik setiap cluster.

In [ ]:
ukuran_cluster = df["Cluster"].value_counts().sort_index()
profil_cluster = df.groupby("Cluster")[fitur_numerik].mean().round(2)
print("Ukuran cluster:")
print(ukuran_cluster)
print("\\nProfil cluster:")
print(profil_cluster)

## 9. Validasi Model dan Pengujian Statistik

Tahap ini menghitung validasi internal cluster dan pengujian statistik untuk mendukung interpretasi hasil.

In [ ]:
# 1. Elbow Method
inertia = []
k_values = range(2, 9)
for k in k_values:
    m = KMeans(n_clusters=k, random_state=42, n_init=10)
    m.fit(X)
    inertia.append(m.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Inertia')
plt.xticks(list(k_values))
plt.tight_layout()
plt.savefig('gambar_elbow_method.png', dpi=150)
plt.show()

In [ ]:
# 2. Silhouette Score, Davies-Bouldin, dan Calinski-Harabasz
validasi = []
for k in k_values:
    m = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = m.fit_predict(X)
    validasi.append({
        'k': k,
        'silhouette_score': silhouette_score(X, labels),
        'davies_bouldin_index': davies_bouldin_score(X.toarray() if hasattr(X, 'toarray') else X, labels),
        'calinski_harabasz_score': calinski_harabasz_score(X.toarray() if hasattr(X, 'toarray') else X, labels)
    })
validasi_df = pd.DataFrame(validasi).round(4)
display(validasi_df)
print('Silhouette terbaik: k =', validasi_df.loc[validasi_df['silhouette_score'].idxmax(), 'k'])
print('Davies-Bouldin terbaik: k =', validasi_df.loc[validasi_df['davies_bouldin_index'].idxmin(), 'k'])
print('Calinski-Harabasz terbaik: k =', validasi_df.loc[validasi_df['calinski_harabasz_score'].idxmax(), 'k'])

In [ ]:
# 3. Kruskal-Wallis untuk Sales, Discount, dan Profit
uji_kw = []
for kolom in ['Sales', 'Discount', 'Profit']:
    kelompok = [g[kolom].dropna().values for _, g in df.groupby('Cluster')]
    statistik, p_value = kruskal(*kelompok)
    uji_kw.append({'variabel': kolom, 'statistik_h': statistik, 'p_value': p_value, 'signifikan_0_05': p_value < 0.05})
uji_kw_df = pd.DataFrame(uji_kw).round(6)
display(uji_kw_df)

In [ ]:
# 4. Korelasi Spearman Discount dan Profit
data_korelasi = df[['Discount', 'Profit']].dropna()
rho, p_value_spearman = spearmanr(data_korelasi['Discount'], data_korelasi['Profit'])
spearman_df = pd.DataFrame([{
    'variabel_x': 'Discount',
    'variabel_y': 'Profit',
    'rho_spearman': rho,
    'p_value': p_value_spearman,
    'signifikan_0_05': p_value_spearman < 0.05
}]).round(6)
display(spearman_df)

In [ ]:
# 5. Chi-Square Category dan Cluster
tabel_kontingensi = pd.crosstab(df['Category'].fillna('Missing'), df['Cluster'])
chi2, p_value_chi, derajat_bebas, expected = chi2_contingency(tabel_kontingensi)
chi_square_df = pd.DataFrame([{
    'chi2': chi2,
    'p_value': p_value_chi,
    'derajat_bebas': derajat_bebas,
    'signifikan_0_05': p_value_chi < 0.05
}]).round(6)
display(tabel_kontingensi)
display(chi_square_df)

## 10. Visualisasi dan Tabel Hasil

Tahap ini menghasilkan gambar, chart, dan tabel yang digunakan pada Bab IV laporan.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Tabel perbandingan sebelum dan sesudah preprocessing
perbandingan = pd.DataFrame({
    'Indikator': ['Jumlah baris', 'Jumlah kolom awal', 'Jumlah missing value fitur model', 'Jumlah duplikat'],
    'Sebelum preprocessing': [len(df_awal), df_awal.shape[1], int(df_awal[fitur].isna().sum().sum()), int(df_awal.duplicated().sum())],
    'Sesudah preprocessing': [len(df), df[fitur].shape[1], 0, 0]
})
display(perbandingan)
perbandingan.to_csv('tabel_perbandingan_preprocessing.csv', index=False)

# Tabel statistik dan profil cluster
display(df[fitur_numerik].describe().round(2))
display(profil_cluster)

# Grafik missing value
missing = df[fitur].isna().sum().sort_values(ascending=False)
plt.figure(figsize=(10, 4))
missing.plot(kind='bar', color='steelblue')
plt.title('Jumlah Missing Value per Fitur')
plt.ylabel('Jumlah')
plt.tight_layout()
plt.savefig('gambar_missing_value.png', dpi=150)
plt.show()

# Distribusi cluster
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='Cluster', color='teal')
plt.title('Jumlah Transaksi per Cluster')
plt.tight_layout()
plt.savefig('gambar_jumlah_cluster.png', dpi=150)
plt.show()

# Scatter Sales dan Profit
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Sales', y='Profit', hue='Cluster', palette='tab10', alpha=0.6)
plt.title('Sales dan Profit Berdasarkan Cluster')
plt.tight_layout()
plt.savefig('gambar_sales_profit_cluster.png', dpi=150)
plt.show()

# Perbandingan Discount dan Profit
profil_plot = df.groupby('Cluster')[['Discount', 'Profit']].mean().reset_index()
profil_long = profil_plot.melt(id_vars='Cluster', var_name='Fitur', value_name='Rata-rata')
plt.figure(figsize=(8, 4))
sns.barplot(data=profil_long, x='Cluster', y='Rata-rata', hue='Fitur')
plt.title('Perbandingan Rata-rata Discount dan Profit')
plt.tight_layout()
plt.savefig('gambar_discount_profit_cluster.png', dpi=150)
plt.show()

## 11. Ekspor Hasil

Dataset, tabel profil, dan gambar hasil analisis disimpan untuk digunakan dalam laporan.

In [ ]:
output_file = "superstore_hasil_preprocessing_clustering.csv"
df.to_csv(output_file, index=False)
profil_cluster.to_csv("tabel_profil_cluster.csv")
files.download(output_file)

## 12. Kesimpulan Implementasi

Implementasi menghasilkan tabel, gambar, dan file CSV. Output aktual harus dibaca dan dijelaskan pada Bab IV laporan berdasarkan hasil notebook yang dijalankan.